In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

from google_play_scraper import app, reviews, Sort 

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

print('Path set. Python will now look in:', os.path.abspath('..'))

Path set. Python will now look in: /home/mine/10 Academy/fintech-review-analytics


In [3]:
from src.scrape import scrape_fintech_reviews, missing_data, duplicate_data, date_normalization, clean_text, modular_nlp_pipeline

print("Function imported successfully")

Function imported successfully


In [4]:
APPS = {
    'cbe': 'com.combanketh.mobilebanking',
    'abbysinya' : 'com.boa.boaMobileBanking',
    'dashen' : 'com.dashen.dashensuperapp',
}

apps_df = scrape_fintech_reviews(APPS)
print(f'\nTotal raw reviews: {len(apps_df)}')

  Scraped 500 reviews for cbe
  Scraped 500 reviews for abbysinya
  Scraped 500 reviews for dashen

Total raw reviews: 1500


In [5]:
for name, pkg in APPS.items():
    try: 
        app_info = app(pkg)
        print(f"\n{name.upper()} APP NAME:{app_info['title']}")
        print(f"        Current Score: {app_info['score']}")
        print(f"        Total Ratings: {app_info['ratings']}")
        print(f"        Total Reviews: {app_info['reviews']}")
        print(f"        Installs: {app_info['installs']}")
    except Exception as e:
        print(f"Error fetching info for {name}: {e}")


CBE APP NAME:Commercial Bank of Ethiopia
        Current Score: 4.1885247
        Total Ratings: 48461
        Total Reviews: 438
        Installs: 5,000,000+

ABBYSINYA APP NAME:BoA Mobile
        Current Score: 3.7
        Total Ratings: 9259
        Total Reviews: 84
        Installs: 1,000,000+

DASHEN APP NAME:Dashen Bank
        Current Score: 4.14
        Total Ratings: 5666
        Total Reviews: 31
        Installs: 1,000,000+


In [6]:
print(f"Shape: {apps_df.shape}")

apps_df.head()

Shape: (1500, 5)


,app,review,rating,date,thumbs
0,cbe,worst,1,2026-05-16 12:15:55,0
1,cbe,this app very full,5,2026-05-16 09:17:00,0
2,cbe,good apps,4,2026-05-16 07:18:33,0
3,cbe,ok,5,2026-05-16 03:43:47,0
4,cbe,this update got crazy i don't know what's goin...,1,2026-05-15 23:20:32,0


In [7]:
#Basic Shape and types
print(f"Total reviews collected: {len(apps_df)}")
print(f"\nColumn dtypes:")
print(apps_df.dtypes)

Total reviews collected: 1500

Column dtypes:
app               object
review            object
rating             int64
date      datetime64[ns]
thumbs             int64
dtype: object


In [8]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(apps_df['date'].head(10).to_string())

print(f"\nDate dtype: {apps_df['date'].dtype}")

Sample date values (raw):
0   2026-05-16 12:15:55
1   2026-05-16 09:17:00
2   2026-05-16 07:18:33
3   2026-05-16 03:43:47
4   2026-05-15 23:20:32
5   2026-05-15 20:11:22
6   2026-05-15 19:53:26
7   2026-05-15 12:22:49
8   2026-05-15 12:07:21
9   2026-05-14 18:52:51

Date dtype: datetime64[ns]


# Data Quality Audit


In [9]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("="*50)

#___ Problem 1: Missing Values ___
print("\nProblem 1: Missing Values")
print("-" * 30)
print(missing_data(apps_df))

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
app            : OK
review         : OK
rating         : OK
date           : OK
thumbs         : OK
None


In [10]:
# Duplicate reviews
print("\nProblem 2: Duplicate")
print("-" * 30)

print(duplicate_data(apps_df))


Problem 2: Duplicate
------------------------------
Exact duplicate reviews: 433
Empty reviews: 0
None


In [11]:
# Date format issues
print("\nProblem 3: Date Format")
print("-" * 30)
print(f"Current dtype: {apps_df['date'].dtype}")
print(f" Sample values: {apps_df['date'].iloc[0]}")
print(f" Target format: YYYY-MM-DD (string or date object)")


Problem 3: Date Format
------------------------------
Current dtype: datetime64[ns]
 Sample values: 2026-05-16 12:15:55
 Target format: YYYY-MM-DD (string or date object)


# Data Cleaning

In [12]:
apps_df = apps_df.copy()

print(f"starting with: {len(apps_df)} reviews")

starting with: 1500 reviews


In [13]:
before = len(apps_df)

critical_cols = ['review', 'rating']
apps_df = apps_df.dropna(subset=critical_cols)

removed = before - len(apps_df)
print(f"Removed {removed} reviews with missing critical data.")
print(f"Remaining: {len(apps_df)}")

Removed 0 reviews with missing critical data.
Remaining: 1500


In [14]:
print("Before Normalization:")
print(apps_df['date'].head(3).to_string())
print(f"dtype: {apps_df['date'].dtype}")

#Convert to pandas datetime then format as YYYY-MM-DD string
print(date_normalization(apps_df))

Before Normalization:
0   2026-05-16 12:15:55
1   2026-05-16 09:17:00
2   2026-05-16 07:18:33
dtype: datetime64[ns]

After Normalization:
0    2026-05-16
1    2026-05-16
2    2026-05-16
dtype: object

Date range: 2025-02-23 to 2026-05-16
None


In [ ]:
#save to CSV
import os

output_path = '../data/processed/review_clean.csv'
apps_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")